# Pristine-anchored model — top-3 cells per task @ 128 bins (GPU)

Trains, at **128-bin** frequency resolution, the **top-3 (model, feature) cells of every task**, on synthetic data from a finite-element model **adjusted as well as possible from the pristine case only** — i.e. with *first-principles, pristine-anchored* damage submodels instead of the lookup tables that were fitted to the damaged experimental FRFs.

**What "pristine-anchored" means here.** The calibrated baseline (stiffness / mass / damping from `calibration_result.npz`) is kept — that *is* the pristine fit. But the damage magnitudes come from `ml_pipeline/pristine_physics.py`, which sizes each mechanism from geometry + textbook mechanics, anchored only at the undamaged state (severity 0 → ratio 1) and **never** from any damaged measurement:

| Damage | Pristine-anchored law | (Calibrated table it replaces) |
|---|---|---|
| Bolt loosening `p%` | per-end `JSR ×= 1 − p/100` (remaining preload) | 11→0.85, 20→0.70, 50→0.55, 85→0.39 |
| Crack `a` mm | `(col_lx − a)/col_lx` (I_xx linear in width) | 5→0.96, 8→0.94 |
| Hole `φ` mm | `1 − (πφ⁴/64)/(col_lx·col_ly³/12)` | 4→0.98, 6→0.97 |
| Mass | kg directly (known test weight) | — (already physical) |

Everything is **additive** — no calibrated module, dataset, notebook or results branch is overwritten. The synthetic set is regenerated to `dataset_pristine/` → `dataset/features_hires_pristine.h5`; results go to the new `colab-hires-pristine128` branch. Experimental features (the measured test set) are reused unchanged.

The top-3 cells per task are picked by their **experimental transfer in the main (calibrated) 128-bin study** (committed `results_hires/per_case_hires128.tar.gz`); those architectures are then re-trained here on the pristine-anchored model, so any drop is attributable to dropping the damaged-data-fitted magnitudes.

**No GPU? Runtime → Change runtime type → GPU.** Private repo: add a Colab secret `GH_TOKEN`.

## 1 · Bootstrap (clone phd_lanl + pymodal, install deps)

In [ ]:
import os, sys, subprocess
GH_USER='grcarmenaty'; WORK='/content'; os.chdir(WORK)
def _tok():
    try:
        from google.colab import userdata; return userdata.get('GH_TOKEN')
    except Exception: return os.environ.get('GH_TOKEN')
def clone(repo, branch, dst):
    if os.path.isdir(dst): print('exists', dst); return
    t=_tok(); auth=f'{t}@' if t else ''
    url=f'https://{auth}github.com/{GH_USER}/{repo}.git'
    assert subprocess.run(['git','clone','--depth','1','-b',branch,url,dst]).returncode==0, \
        f'clone failed {repo}@{branch} (private? add a GH_TOKEN Colab secret)'
clone('phd_lanl','main','/content/PhD_LANL')
clone('pymodal','master','/content/pymodal')   # sibling dir the scripts expect
for p in ('/content/PhD_LANL','/content/pymodal'):
    if p not in sys.path: sys.path.insert(0,p)
os.chdir('/content/PhD_LANL')
# Harden git's HTTP transport against Drive-mounted-Colab flakiness (the 408s):
for _k,_v in [('http.postBuffer','524288000'),('http.version','HTTP/1.1'),
              ('http.lowSpeedLimit','1000'),('http.lowSpeedTime','300')]:
    subprocess.run(['git','config','--global',_k,_v])
subprocess.run([sys.executable,'-m','pip','-q','install','timm','h5py','scikit-learn','xgboost','pint','pyFRF','audiomentations'])
import torch
print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),'|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - set a GPU runtime!')

## 2 · Build the PRISTINE-anchored synthetic features (+ experimental features)

In [ ]:
import subprocess, sys, os, glob, json, h5py, numpy as np
from pathlib import Path
REPO=Path(os.getcwd())
def run(cmd): print('>>',' '.join(cmd)); assert subprocess.run(cmd).returncode==0, cmd
# --- Synthetic = FE model with FIRST-PRINCIPLES, pristine-anchored damage ---
#     (generate_dataset_pristine.py swaps the damage submodels; nothing else)
if not (REPO/'dataset'/'features_hires_pristine.h5').exists():
    run([sys.executable,'ml_pipeline/generate_dataset_pristine.py','--out','dataset_pristine','--n-t','4096','--fs','256'])
    run([sys.executable,'ml_pipeline/build_hires_synth_features.py',
         '--chunks','dataset_pristine','--out','dataset/features_hires_pristine.h5'])
# --- Experimental = the real measurements (test set), unchanged ---
if not (REPO/'experimental_frfs.h5').exists():
    with open('experimental_frfs.h5','wb') as o:
        for p in sorted(glob.glob('experimental_frfs_chunks/experimental_frfs.h5.part_*')):
            o.write(open(p,'rb').read())
if not (REPO/'dataset'/'experimental_features.h5').exists():
    from ml_pipeline.evaluate import primary_op
    with h5py.File('experimental_frfs.h5','r') as f: names=json.loads(f.attrs['case_names_json'])
    n=len(names); tc=np.zeros(n,np.int8); st=np.full(n,-1,np.int8); en=np.full(n,-1,np.int8); sv=np.zeros(n,np.float32)
    for i,nm in enumerate(names):
        op=primary_op(nm); tc[i]=op['type_code']; st[i]=op['storey']; en[i]=op['end']; sv[i]=op['severity']
    dt=h5py.string_dtype('utf-8')
    with h5py.File('dataset/experimental_features.h5','w') as o:
        o.create_dataset('names',data=np.array(names,dtype=object),dtype=dt)
        o.create_dataset('type_code',data=tc); o.create_dataset('storey',data=st)
        o.create_dataset('end',data=en); o.create_dataset('severity',data=sv)
if not (REPO/'dataset'/'experimental_features_hires.h5').exists():
    run([sys.executable,'ml_pipeline/build_hires_exp_features.py'])
print('features ready (synth = PRISTINE-anchored model, exp = measured)')

## 3 · Pick the top-3 cells per task

Ranked by **experimental balanced-accuracy** (classification) / **R²** (severity) in the committed main-study 128-bin results. Collapsed cells sink to the bottom; we keep the 3 best distinct `(model, feature)` per task.

In [ ]:
import tarfile, tempfile, json, glob, os
from collections import defaultdict
from sklearn.metrics import balanced_accuracy_score, r2_score
TASKS=['binary','col_location','mass_location','severity','type',
       'is_bolt','is_crack','is_mass','is_hole','is_pristine']
TOPK=3
_tmp=tempfile.mkdtemp()
with tarfile.open('results_hires/per_case_hires128.tar.gz') as tf: tf.extractall(_tmp)
scores=defaultdict(list)
for p in glob.glob(os.path.join(_tmp,'**','*_hires128.json'),recursive=True):
    d=json.load(open(p)); m=d['meta']; r=d['rows']; t=m['task']
    if t not in TASKS: continue
    yt=[x['y_true'] for x in r]; yp=[x['y_pred'] for x in r]
    if m['kind']=='cls':
        s=balanced_accuracy_score(yt,yp); coll=len(set(yp))<=1
    else:
        s=r2_score([float(a) for a in yt],[float(a) for a in yp]); coll=False
    scores[t].append((s,m['model'],m['feature'],m['kind'],coll))
TOP3={}; CELLS=[]
for t in TASKS:
    seen=set(); pick=[]
    for s,mo,ft,kind,coll in sorted(scores[t], key=lambda z:(z[4], -z[0])):
        if (mo,ft) in seen: continue
        seen.add((mo,ft)); pick.append((mo,ft,s,kind))
        if len(pick)>=TOPK: break
    TOP3[t]=pick
    for mo,ft,s,kind in pick: CELLS.append((t,mo,ft))
    print(f"{t:14s} "+" | ".join(f"{mo}/{ft} {s:.3f}" for mo,ft,s,kind in pick))
print(f"\n{len(CELLS)} cells queued (top-{TOPK} per task)")

## 4 · Config + context + tabular caches

In [ ]:
import torch, numpy as np, h5py
from pathlib import Path
from ml_pipeline import hires_zoo as Z
from ml_pipeline import hires_tab as T
from ml_pipeline.tasks import build_targets
from ml_pipeline.train import make_split
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ===================== CONFIG (edit me) =====================
RES        = 128            # frequency resolution (bins) for this study
SUBSAMPLE  = 4000           # synth samples per cell
BATCH_VIS  = 16             # vision / CFDAC-CNN batch (heavy)
BATCH_TAB  = 256            # tabular / sequence batch
MAX_EPOCHS = 80; PATIENCE = 8; VISION_SIZE = 384
FAMILY            = 'pristine128'
AUTOSAVE_GITHUB   = True
GH_RESULTS_BRANCH = 'colab-hires-pristine128'   # never touches main / other families
# To run ONE cell this session, e.g.:  CELLS = [('is_hole','convnext_tiny','cfdac_all')]
print(len(CELLS),'cells across models', sorted({m for _,m,_ in CELLS}))
# ===========================================================

try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT=Path('/content/drive/MyDrive/hires_cfdac/pristine128')
except Exception:
    OUT=Path('results_hires_zoo_pristine128')
OUT.mkdir(parents=True, exist_ok=True); (OUT/'cache').mkdir(exist_ok=True); print('OUT =', OUT)

# Resume: seed OUT/per_case from the results branch so finished cells are skipped.
import subprocess as _sp, os as _os
try:
    _sp.run(['git','-C','/content/PhD_LANL','fetch','--depth','1','origin',GH_RESULTS_BRANCH], capture_output=True)
    _ls=_sp.run(['git','-C','/content/PhD_LANL','ls-tree','-r','--name-only','FETCH_HEAD'],capture_output=True,text=True).stdout
    (OUT/'per_case').mkdir(parents=True, exist_ok=True); _n=0
    for _l in _ls.splitlines():
        if f'results_hires_zoo/{FAMILY}/per_case/' in _l and _l.endswith('.json'):
            _name=_os.path.basename(_l)
            if not (OUT/'per_case'/_name).exists():
                _b=_sp.run(['git','-C','/content/PhD_LANL','show','FETCH_HEAD:'+_l],capture_output=True,text=True).stdout
                if _b: (OUT/'per_case'/_name).write_text(_b); _n+=1
    print('picked up',_n,'already-trained cells from',GH_RESULTS_BRANCH)
except Exception as _e: print('branch pickup skipped:', _e)

# Synthetic = PRISTINE-anchored model;  Experimental = measured (unchanged).
SYN=Path('dataset/features_hires_pristine.h5'); EXP=Path('dataset/experimental_features_hires.h5')
with h5py.File(SYN,'r') as f:
    syn_tasks=build_targets(f['type_code'][:].astype('int64'),f['storey'][:].astype('int64'),
                            f['end'][:].astype('int64'),f['severity'][:].astype('float32'))
    H_ref_syn=torch.from_numpy(f['reference/frf_complex'][:].astype('complex64')).to(DEV)
with h5py.File(EXP,'r') as f:
    exp_tasks=build_targets(f['type_code'][:].astype('int64'),f['storey'][:].astype('int64'),
                            f['end'][:].astype('int64'),f['severity'][:].astype('float32'))
    exp_names=[str(s) for s in f['names'][:]]
    H_ref_exp=torch.from_numpy(f['reference/frf_complex'][:].astype('complex64')).to(DEV)
    H_exp=(f['frf_real'][:]+1j*f['frf_imag'][:]).astype('complex64')

# Cache any non-CFDAC (tabular/sequence) features once, at RES, for syn + exp.
TABCACHE={}
for ft in sorted({f for (_,_,f) in CELLS if not f.startswith('cfdac')}):
    Xs=T.build_feature_cache(str(SYN), ft, OUT/'cache'/f'{ft}_syn_{RES}.npy', res=RES)
    Xe=T.build_feature_cache(str(EXP), ft, OUT/'cache'/f'{ft}_exp_{RES}.npy', res=RES)
    TABCACHE[ft]=(Xs,Xe)
print('context ready; exp FRFs', H_exp.shape, '| tab caches', list(TABCACHE))
print('device', DEV, '| amp dtype', Z._amp_dtype(DEV))

## 5 · Train the selected cells on the pristine-anchored model

In [ ]:
import torch, os, shutil, subprocess
def _tok():
    try:
        from google.colab import userdata; return userdata.get('GH_TOKEN')
    except Exception: return os.environ.get('GH_TOKEN')
GH_TOKEN=_tok()
if AUTOSAVE_GITHUB and not GH_TOKEN:
    print('AUTOSAVE on but no GH_TOKEN -> Drive/zip only')

def git_autosave(msg):
    if not (AUTOSAVE_GITHUB and GH_TOKEN): return
    repo='/content/PhD_LANL'; dst=os.path.join(repo,'results_hires_zoo',FAMILY)
    os.makedirs(os.path.join(dst,'per_case'), exist_ok=True)
    if not getattr(git_autosave,'_merged',False):   # one-time: merge remote cells so a force-push never drops work
        subprocess.run(['git','-C',repo,'fetch','--depth','1','origin',GH_RESULTS_BRANCH],capture_output=True)
        _rl=subprocess.run(['git','-C',repo,'ls-tree','-r','--name-only','FETCH_HEAD'],capture_output=True,text=True).stdout
        for _l in _rl.splitlines():
            if '/per_case/' in _l and _l.endswith('.json'):
                _fp=os.path.join(str(OUT),'per_case',os.path.basename(_l))
                if not os.path.exists(_fp):
                    _bb=subprocess.run(['git','-C',repo,'show','FETCH_HEAD:'+_l],capture_output=True,text=True).stdout
                    if _bb: open(_fp,'w').write(_bb)
        git_autosave._merged=True
    for fn in (os.listdir(os.path.join(OUT,'per_case')) if os.path.isdir(os.path.join(OUT,'per_case')) else []):
        if fn.endswith('.json'): shutil.copy(os.path.join(OUT,'per_case',fn), os.path.join(dst,'per_case',fn))
    for sj in ('synth_test_zoo.json','synth_test_tab.json'):
        if os.path.exists(os.path.join(OUT,sj)): shutil.copy(os.path.join(OUT,sj), os.path.join(dst,sj))
    cwd=os.getcwd(); os.chdir(repo)
    subprocess.run(['git','config','user.email','colab@gpu.run']); subprocess.run(['git','config','user.name','colab-gpu'])
    subprocess.run(['git','add','-f',f'results_hires_zoo/{FAMILY}'])
    if subprocess.run(['git','diff','--cached','--quiet']).returncode!=0:
        subprocess.run(['git','commit','-q','-m',msg])
        url=f'https://{GH_TOKEN}@github.com/grcarmenaty/phd_lanl.git'
        import time as _t; ok=False
        for _a in range(5):
            r=subprocess.run(['git','push','--force',url,f'HEAD:{GH_RESULTS_BRANCH}'],capture_output=True,text=True)
            if r.returncode==0: ok=True; break
            _t.sleep(4*(2**_a))
        print('  autosave:', f'pushed -> {GH_RESULTS_BRANCH}' if ok else 'push failed (Drive has results): '+r.stderr[-140:])
    os.chdir(cwd)

for (task, model, feature) in CELLS:
    try:
        if feature.startswith('cfdac'):           # vision / CFDAC-CNN path
            Z.run_cell(task, model, feature, syn_h5=SYN, exp_h5=EXP, out_dir=OUT, dev=DEV,
                       syn_tasks=syn_tasks, exp_tasks=exp_tasks, H_ref_syn=H_ref_syn,
                       H_ref_exp=H_ref_exp, H_exp=H_exp, exp_names=exp_names,
                       make_split=make_split, subsample=SUBSAMPLE, batch=BATCH_VIS,
                       vision_size=VISION_SIZE, max_epochs=MAX_EPOCHS, patience=PATIENCE, res_bins=RES)
        else:                                      # tabular / sequence path
            Xs,Xe=TABCACHE[feature]
            T.run_tab_cell(task, model, feature, out_dir=OUT, dev=DEV, syn_tasks=syn_tasks,
                           exp_tasks=exp_tasks, Xsyn=Xs, Xexp=Xe, exp_names=exp_names,
                           make_split=make_split, subsample=SUBSAMPLE, batch=BATCH_TAB, res=RES)
        git_autosave(f'colab autosave [{FAMILY}]: {task}/{model}/{feature}')
    except Exception as e:
        print('CELL FAILED', task, model, feature, '::', repr(e)[:200])
        if torch.cuda.is_available(): torch.cuda.empty_cache()
print('\nqueue done')

## 6 · Honest summary (synth vs experimental transfer) + zip

In [ ]:
import json, numpy as np
from pathlib import Path
from sklearn.metrics import balanced_accuracy_score, f1_score, r2_score
print(f"{'cell':<48}{'kind':>5}{'synth':>8}{'expMF1/R2':>11}{'expBal':>8}{'collapse':>9}")
print('-'*88)
for p in sorted((OUT/'per_case').glob(f'*_hires{RES}.json')):
    d=json.loads(p.read_text()); m=d['meta']; r=d['rows']
    yt=np.array([x['y_true'] for x in r]); yp=np.array([x['y_pred'] for x in r])
    name=f"{m['task']}/{m['model']}/{m['feature']}"
    if m['kind']=='cls':
        n=m['n_out']; bal=balanced_accuracy_score(yt,yp)
        mf1=f1_score(yt,yp,labels=list(range(n)),average='macro',zero_division=0)
        coll=(len(set(yp.tolist()))<=1) or (bal<=1/n+0.02)
        print(f"{name:<48}{'cls':>5}{(m.get('synth_test_macro_f1') or 0):>8.3f}{mf1:>11.3f}{bal:>8.3f}{str(coll):>9}")
    else:
        yt=yt.astype(float); yp=yp.astype(float); r2=r2_score(yt,yp) if np.var(yt)>0 else 0.0
        print(f"{name:<48}{'reg':>5}{(m.get('synth_test_metric') or 0):>8.3f}{r2:>11.3f}{'-':>8}{'-':>9}")
import shutil
z=str(OUT).rstrip('/').split('/')[-1]
shutil.make_archive('/content/'+z,'zip',str(OUT))
try:
    from google.colab import files; files.download('/content/'+z+'.zip')
except Exception as e: print('zip at /content/'+z+'.zip', e)

## 7 · (Optional) push the JSON snapshot to the results branch

In [ ]:
import os, subprocess, shutil, glob, time as _t
tok=None
try:
    from google.colab import userdata; tok=userdata.get('GH_TOKEN')
except Exception: tok=os.environ.get('GH_TOKEN')
if not tok:
    print('No GH_TOKEN - download the zip from the cell above and hand it to the agent.')
else:
    repo='/content/PhD_LANL'; dst=os.path.join(repo,'results_hires_zoo',FAMILY)
    os.makedirs(os.path.join(dst,'per_case'), exist_ok=True)
    for fn in (os.listdir(os.path.join(OUT,'per_case')) if os.path.isdir(os.path.join(OUT,'per_case')) else []):
        if fn.endswith('.json'): shutil.copy(os.path.join(OUT,'per_case',fn), os.path.join(dst,'per_case',fn))
    for sj in glob.glob(os.path.join(OUT,'synth_test_*.json')):
        shutil.copy(sj, os.path.join(dst, os.path.basename(sj)))
    os.chdir(repo)
    subprocess.run(['git','config','user.email','colab@gpu.run']); subprocess.run(['git','config','user.name','colab-gpu'])
    subprocess.run(['git','add','-f',f'results_hires_zoo/{FAMILY}'])
    subprocess.run(['git','commit','-q','-m',f'hires {FAMILY} (GPU): manual JSON snapshot'])
    url=f'https://{tok}@github.com/grcarmenaty/phd_lanl.git'; ok=False
    for _a in range(5):
        r=subprocess.run(['git','push','--force',url,f'HEAD:{GH_RESULTS_BRANCH}'],capture_output=True,text=True)
        if r.returncode==0: ok=True; break
        _t.sleep(4*(2**_a))
    print(f'pushed -> {GH_RESULTS_BRANCH}' if ok else 'push failed after retries: '+r.stderr[-200:])